# FPF04 — BOZORTH3 on FingerJetFX Templates: Minutiae Verification Baseline on the FPF02 Protocol (3 repetitions)

**Purpose:** the classic reference for the paper, as FPM03 but on the new data and the FPF02 protocol: NIST NBIS **BOZORTH3** matching the **FingerJetFX** templates made by FPF00, in 1:1 verification, on exactly the same pairs as FPF02 (and FPF05).

* **Pairs:** every FPF02 pair set of every repetition — training (DB1, DB2, DB1×DB2; used only to fix the threshold), validation and test (DB1, DB2, DB1×DB2 cross-sensor) and **FingerDOS** (external test). Same subjects, same pairs, same seeds as FPF02.
* **Matcher:** BOZORTH3 compiled from a pinned NBIS commit, ANSI/ISO `-m1` mode, 150 highest-quality minutiae per template (its default), as FPM03. A template with no minutia (FingerJetFX failure) scores 0.
* **Threshold (agreed for the paper):** one frozen threshold = the **EER point of the training pairs** (DB1, DB2 and DB1×DB2 weighted equally), raw BOZORTH3 scores, then applied unchanged to validation, test and FingerDOS. No FMR ≤ 1% rule.
* **Metrics:** AUC, EER, FMR, FNMR, F1-macro (+ TAR@FMR 1%, balanced accuracy, precision), per set and pooled (MOLF validation, MOLF test, MOLF same-sensor test), mean ± std over repetitions.
* **Time:** FingerJetFX extraction per image (from FPF00), template → xyt conversion, BOZORTH3 per pair (single thread, 2,000 pairs per process) and per single call, template size, and the resulting 1:1 verification time from an image.
* **Comparison:** the FPF02 image model, re-evaluated from its saved scores with the same threshold rule on the identical pairs (per-repetition paired differences).

**Hardware:** CPU (more cores = faster), High-RAM. ≈ 130,000 pairs per repetition; identical pairs are scored once and cached. Finished repetitions are skipped.

In [ ]:
%pip install -q openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the shared FPF library written by FPF00 (output/00_code/), so every FPF kernel uses the same version.
import shutil
from pathlib import Path
def _find_folder(name, root=Path('/content/drive/MyDrive'), max_depth=4):
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: kids = [c for c in d.iterdir() if c.is_dir()]
            except OSError: continue
            for c in kids:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None
ROOT = _find_folder('FingerPrint-Final')
if ROOT is None: raise FileNotFoundError("'FingerPrint-Final' not found under My Drive")
for m in ('fpf_lib.py',):
    src = ROOT / 'output' / '00_code' / m
    if not src.is_file(): raise FileNotFoundError(f'{src} missing: run FPF00 first')
    shutil.copy2(src, Path('/content') / m)
import sys, importlib; sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import numpy as np, pandas as pd, json, time, os, gc
ENV = L.env_info(); print(json.dumps(ENV, indent=1))

## 1. Configuration and protocol

In [ ]:
import subprocess, tempfile, h5py
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
KERNEL = 'FPF04'; KERNEL_NAME = 'FPF04_BOZORTH3_FingerJetFX_Minutiae_Verification_Baseline_DB1DB2_FingerDOSTest'; ARM = 'BOZORTH3'
ws = L.workspace(); KDIR = ws['output'] / KERNEL_NAME; FIG = KDIR / 'figures'
for _d in (FIG, KDIR / 'runs'): _d.mkdir(parents=True, exist_ok=True)
LOG = L.RunLog(KDIR / 'FPF04_run_log.csv'); REPS = L.repetitions(ws)
NBIS_REPO = 'https://github.com/mike10004/nbis.git'; NBIS_COMMIT = 'e2122d4508ab5cc4761fe29caa8a161539c177d8'
MAX_MINUTIAE = 150; WORKERS = max(2, os.cpu_count() or 2); WORK = Path('/content/fpf04_work'); WORK.mkdir(exist_ok=True)
TIMING_PAIRS = 2000; SINGLE_CALLS = 50
CONFIG = dict(kernel=KERNEL, kernel_name=KERNEL_NAME, matcher='NIST NBIS BOZORTH3', nbis_repo=NBIS_REPO, nbis_commit=NBIS_COMMIT, mode='-m1 (ANSI/ISO convention)', max_minutiae=MAX_MINUTIAE,
              templates='FPF00 FingerJetFX ISO 19794-2 templates (native resolution)', empty_template_score=0, protocol='FPF02 (identical subject splits and pair files)',
              threshold='EER on the DB1 + DB2 + DB1xDB2 training pairs (sources weighted equally), raw scores; frozen for validation, test and FingerDOS', repetitions=REPS)
print(json.dumps(CONFIG, indent=2))

In [ ]:
# ---- FPF02 protocol: the identical subject splits and pair files for every paper kernel ----
F02_NAME = 'FPF02_MobileNetV3Large_Siamese_FullFT_MildAug_DB1DB2JointTrain_CrossSensorPairs_FMR1pctThreshold_FingerDOSTest'; F02DIR = ws['output'] / F02_NAME
PAIR_SETS = {   # name: (dataset of image a, dataset of image b, partition)
    'DB1_train': ('MOLF_DB1', 'MOLF_DB1', 'train'), 'DB2_train': ('MOLF_DB2', 'MOLF_DB2', 'train'), 'DB1xDB2_train': ('MOLF_DB1', 'MOLF_DB2', 'train'),
    'DB1_validation': ('MOLF_DB1', 'MOLF_DB1', 'validation'), 'DB2_validation': ('MOLF_DB2', 'MOLF_DB2', 'validation'), 'DB1xDB2_validation': ('MOLF_DB1', 'MOLF_DB2', 'validation'),
    'DB1_test': ('MOLF_DB1', 'MOLF_DB1', 'test'), 'DB2_test': ('MOLF_DB2', 'MOLF_DB2', 'test'), 'DB1xDB2_test': ('MOLF_DB1', 'MOLF_DB2', 'test'),
    'FingerDOS_test': ('FingerDOS', 'FingerDOS', 'external_test')}
FPF02_PAIR_FILES = ('DB2_train', 'DB1xDB2_train', 'DB2_validation', 'DB1xDB2_validation')     # built by FPF02 section 2; every other set is the FPF00 file
TRAIN_SOURCES = ('DB1_train', 'DB2_train', 'DB1xDB2_train'); VAL_SOURCES = ('DB1_validation', 'DB2_validation', 'DB1xDB2_validation')
MOLF_TESTS = ('DB1_test', 'DB2_test', 'DB1xDB2_test'); TEST_SETS = MOLF_TESTS + ('FingerDOS_test',)
GROUPS = {'MOLF validation (pooled)': VAL_SOURCES, 'MOLF test (pooled)': MOLF_TESTS, 'MOLF test same-sensor (pooled)': ('DB1_test', 'DB2_test')}
ROLE = {**{n: 'training (threshold only)' for n in TRAIN_SOURCES}, **{n: 'validation' for n in VAL_SOURCES}, **{n: 'test' for n in MOLF_TESTS}, 'FingerDOS_test': 'external test',
        'MOLF validation (pooled)': 'validation', 'MOLF test (pooled)': 'test', 'MOLF test same-sensor (pooled)': 'test'}
REPORT_ORDER = ['DB1_validation', 'DB2_validation', 'DB1xDB2_validation', 'MOLF validation (pooled)', 'DB1_test', 'DB2_test', 'DB1xDB2_test', 'MOLF test (pooled)',
                'MOLF test same-sensor (pooled)', 'FingerDOS_test']
PAPER_METRICS = ['roc_auc', 'eer', 'fmr', 'fnmr', 'f1_macro']; EXTRA_METRICS = ['tar_at_fmr_1pct', 'balanced_accuracy', 'precision']
THRESHOLD_RULE = 'train_EER (DB1, DB2, DB1xDB2 training pairs weighted equally; raw scores, no centering)'

def load_pairs(rep):
    proto = L.load_protocol(ws, rep); P = {n: proto['pairs'][n] for n in PAIR_SETS if n not in FPF02_PAIR_FILES}
    for n in FPF02_PAIR_FILES:
        f = F02DIR / 'protocol' / rep / 'pairs' / f'{n}_pairs.npz'
        if not f.is_file(): raise FileNotFoundError(f'{f} missing: run FPF02 section 2 first (it builds the joint DB1 + DB2 pair files)')
        P[n] = L.load_npz(f)
    return proto, P

def eer_threshold_balanced(ys, ss):
    # EER threshold on the pooled training sources, each source's genuine and impostor pairs weighted equally (the agreed paper threshold)
    y = np.concatenate(ys); s = np.concatenate(ss); w = np.concatenate([np.where(yy == 1, 1.0 / max(yy.sum(), 1), 1.0 / max((yy == 0).sum(), 1)) for yy in ys])
    fpr, tpr, thr = roc_curve(y, s, sample_weight=w, drop_intermediate=False); i = int(np.nanargmin(np.abs(fpr - (1 - tpr)))); return float(thr[i])

def evaluate(S, kernel, arm, rep, extra=None):
    # S: {pair_set: (label, score)} for all 10 sets. One frozen threshold from the training pairs; every other set and pooled group uses it unchanged.
    thr = eer_threshold_balanced([S[n][0] for n in TRAIN_SOURCES], [S[n][1] for n in TRAIN_SOURCES]); rows = []
    def add(name, y, s, members):
        m = L.verification_metrics(y, s, thr)
        m.update({'kernel': kernel, 'arm': arm, 'repetition': rep, 'set': name, 'members': ' + '.join(members), 'role': ROLE[name], 'threshold_rule': 'train_EER', **(extra or {})}); rows.append(m)
    for n in PAIR_SETS: add(n, *S[n], [n])
    for g, mem in GROUPS.items(): add(g, np.concatenate([S[n][0] for n in mem]), np.concatenate([S[n][1] for n in mem]), mem)
    return rows, thr

def save_scores(S, rd):
    np.savez_compressed(Path(rd) / 'pair_scores.npz', **{f'{n}__raw__score': np.asarray(s, np.float32) for n, (y, s) in S.items()}, **{f'{n}__label': np.asarray(y, np.int8) for n, (y, s) in S.items()})
def load_scores(rd):
    z = np.load(Path(rd) / 'pair_scores.npz'); return {n: (z[f'{n}__label'].astype(int), z[f'{n}__raw__score'].astype(np.float64)) for n in PAIR_SETS if f'{n}__raw__score' in z.files}

def reference_rows(label, kdir, arm=None, reps=None):
    # re-evaluates another paper kernel from its saved pair scores with the same threshold rule (identical pairs); [] if it has not run
    kdir = Path(kdir)
    if arm is None:
        if not (kdir / 'winner_manifest.json').is_file(): print(f'{label}: not found ({kdir.name}) -> skipped'); return []
        arm = json.loads((kdir / 'winner_manifest.json').read_text())['winning_arm']
    rows = []
    for r in reps or REPS:
        rd = kdir / 'runs' / r / arm
        if not (rd / 'pair_scores.npz').is_file(): print(f'{label}: {r} missing -> skipped'); continue
        S = load_scores(rd)
        if set(S) != set(PAIR_SETS): print(f'{label}: {r} lacks some pair sets -> skipped'); continue
        rows += evaluate(S, label, arm, r)[0]
    return rows

def paper_table(results, models=None):
    s = L.summarize(results, ['kernel', 'set']); s = s[s.set.isin(REPORT_ORDER)].copy()
    s['set'] = pd.Categorical(s.set, REPORT_ORDER); s['kernel'] = pd.Categorical(s.kernel, models or list(dict.fromkeys(results.kernel)))
    s = s.sort_values(['set', 'kernel']); out = s[['set', 'kernel']].copy(); out.insert(1, 'role', [ROLE[x] for x in s.set])
    for m in PAPER_METRICS + EXTRA_METRICS: out[m] = [f'{a:.2f} ± {0 if np.isnan(b) else b:.2f}' for a, b in zip(s[f'{m}_mean'], s[f'{m}_std'])]
    return out.rename(columns={'roc_auc': 'AUC ↑', 'eer': 'EER ↓', 'fmr': 'FMR ↓', 'fnmr': 'FNMR ↓', 'f1_macro': 'F1 ↑', 'tar_at_fmr_1pct': 'TAR@FMR1% ↑',
                               'balanced_accuracy': 'Bal. acc. ↑', 'precision': 'Precision ↑'}), s

def paired_differences(results, cur, refs):
    # per repetition, same pairs: cur minus ref (points); same_sign_all_reps marks a consistent direction
    key = ['repetition', 'set']; c = results[results.kernel == cur].set_index(key); out = []
    for ref in refs:
        b = results[results.kernel == ref].set_index(key)
        for m in PAPER_METRICS + ['tar_at_fmr_1pct']:
            d = (100 * (c[m] - b[m])).dropna()
            for st, g in d.groupby(level='set'):
                if st not in REPORT_ORDER: continue
                out.append({'model': cur, 'vs': ref, 'set': st, 'metric': m, 'mean_difference_points': g.mean(), 'std': g.std(ddof=1), 'min': g.min(), 'max': g.max(),
                            'repetitions': len(g), 'same_sign_all_reps': bool((g > 0).all() or (g < 0).all())})
    return pd.DataFrame(out)

def fig_model_bars(snum, models, path, sets=('MOLF validation (pooled)', 'MOLF test (pooled)', 'DB1xDB2_test', 'FingerDOS_test')):
    fig, axs = plt.subplots(1, 5, figsize=(23, 4.2)); x = np.arange(len(sets)); w = 0.8 / len(models)
    for ax, m, lab in zip(axs, PAPER_METRICS, ('AUC (%) ↑', 'EER (%) ↓', 'FMR (%) ↓', 'FNMR (%) ↓', 'F1-macro (%) ↑')):
        for k, mod in enumerate(models):
            q = snum[snum.kernel == mod].set_index('set').reindex(list(sets))
            ax.bar(x + (k - (len(models) - 1) / 2) * w, q[f'{m}_mean'], w, yerr=q[f'{m}_std'].fillna(0), capsize=2, color=L.COLORS[k % 10], label=mod)
        ax.set(xticks=x, ylabel=lab); ax.set_xticklabels([s.replace(' (pooled)', '\n(pooled)') for s in sets], fontsize=7); ax.grid(alpha=.3, axis='y')
    axs[0].legend(fontsize=7); fig.suptitle('Frozen training-EER threshold, mean ± std over repetitions (identical pairs)', fontsize=10); L._save(fig, path)
print('FPF02 protocol:', len(PAIR_SETS), 'pair sets | threshold:', THRESHOLD_RULE)

## 2. Build BOZORTH3 from the pinned NBIS source (direct gcc build, as FPM03)

In [ ]:
def run(cmd, cwd=None):
    p = subprocess.run(cmd, cwd=cwd, shell=True, text=True, capture_output=True)
    if p.returncode != 0: raise RuntimeError(f'{cmd}\n{p.stdout[-2000:]}\n{p.stderr[-2000:]}')
    return p.stdout
SRC = Path('/content/nbis_src')
if not (SRC / '.git').is_dir(): run(f'git clone -q {NBIS_REPO} {SRC}')
run(f'git checkout -q {NBIS_COMMIT}', cwd=SRC); BZ = WORK / 'bozorth3_build'; BZ.mkdir(exist_ok=True)
(BZ / 'version.h').write_text('#define VERSION "NBIS bozorth3 (FPF04 direct build)"\n')
(BZ / 'version_stub.c').write_text('#include <stdio.h>\nvoid getVersion(void){printf("bozorth3 (FPF04 direct build)\\n");}\n')
BOZORTH3 = BZ / 'bozorth3'
run(f'gcc -O2 -w -I{BZ} -Ibozorth3/include -Icommonnbis/include bozorth3/src/bin/bozorth3/bozorth3.c bozorth3/src/bin/bozorth3/usage.c bozorth3/src/lib/bozorth3/*.c {BZ}/version_stub.c -o {BOZORTH3} -lm', cwd=SRC)
print('Built', BOZORTH3); LOG.add(step='build_bozorth3', status='ok')

## 3. Convert every FingerJetFX template to xyt once

In [ ]:
def write_xyt(path, x, y, theta_img_deg, quality):
    # FPF00 stores theta in the image convention (360 - ISO angle); BOZORTH3 -m1 takes ISO angles: convert back, keep the MAX_MINUTIAE best-quality minutiae
    iso = np.rint((360.0 - np.asarray(theta_img_deg, np.float64)) % 360.0).astype(int) % 360; q = np.rint(100 * np.asarray(quality)).astype(int)
    with open(path, 'w') as f:
        for i in np.argsort(-q, kind='stable')[:MAX_MINUTIAE]: f.write(f'{int(round(float(x[i])))} {int(round(float(y[i])))} {iso[i]} {q[i]}\n')
def bozorth3_scores(pairs_of_paths, chunk=2000):
    out = []
    for s in range(0, len(pairs_of_paths), chunk):
        part = pairs_of_paths[s:s + chunk]
        with tempfile.NamedTemporaryFile('w', suffix='.lis', delete=False) as f:
            for a, b in part: f.write(f'{a}\n{b}\n')
            lis = f.name
        p = subprocess.run([str(BOZORTH3), '-m1', '-M', lis], text=True, capture_output=True); os.unlink(lis)
        if p.returncode != 0: raise RuntimeError(p.stderr[-800:])
        v = [int(t) for t in p.stdout.split()]
        if len(v) != len(part): raise RuntimeError('BOZORTH3 returned the wrong number of scores')
        out.extend(v)
    return np.asarray(out, np.float64)
XYT, NMIN, CONV_MS, TEMPLATE_BYTES, XYT_BYTES, EXTRACT_MS = {}, {}, {}, {}, {}, {}
for d in L.DATASETS:
    (WORK / 'xyt' / d).mkdir(parents=True, exist_ok=True)
    with h5py.File(L.local_copy(ws['templates'] / f'{d}_fingerjetfx_templates.h5'), 'r') as h:
        if 'complete' in h and not h['complete'][:].all(): raise RuntimeError(f'{d}: FingerJetFX templates incomplete; finish FPF00')
        g = h['minutiae']; X, Y, TH, Q = g['x'][:], g['y'][:], g['theta'][:], g['quality'][:]
        TEMPLATE_BYTES[d] = float(np.mean([len(v) for v in g['iso_template'][:]])); EXTRACT_MS[d] = float(np.mean(g['extract_ms'][:])) if 'extract_ms' in g else float('nan')
    paths = [WORK / 'xyt' / d / f'{i:06d}.xyt' for i in range(len(X))]; t = L.timer()
    for i, p in enumerate(tqdm(paths, desc=f'{d} xyt')): write_xyt(p, X[i], Y[i], TH[i], Q[i])
    XYT[d] = paths; NMIN[d] = np.array([min(len(v), MAX_MINUTIAE) for v in X]); CONV_MS[d] = 1000 * t() / len(paths); XYT_BYTES[d] = float(np.mean([p.stat().st_size for p in paths]))
    print(f'{d}: {len(paths)} templates, {int((NMIN[d] == 0).sum())} without minutiae, median {int(np.median(NMIN[d]))} minutiae')
_i = int(np.flatnonzero(NMIN['MOLF_DB1'] > 10)[0]); _j = int(np.flatnonzero(NMIN['MOLF_DB1'] > 10)[-1])
self_score = bozorth3_scores([(XYT['MOLF_DB1'][_i], XYT['MOLF_DB1'][_i]), (XYT['MOLF_DB1'][_i], XYT['MOLF_DB1'][_j])]); print('Self-test (same template, different finger):', self_score)
if not self_score[0] > self_score[1]: raise RuntimeError('BOZORTH3 self-test failed')
LOG.add(step='xyt', status='ok', **{f'empty_{d}': int((NMIN[d] == 0).sum()) for d in L.DATASETS})

## 4. Score every FPF02 pair set, every repetition (resumable; identical pairs scored once)

In [ ]:
CACHE = {}
def score_pairs(da, db, P):
    ia, ib = np.asarray(P['index_a']), np.asarray(P['index_b']); out = np.zeros(len(ia), np.float64)
    keys = list(zip(ia.tolist(), ib.tolist())); todo = [k for k in dict.fromkeys(keys) if (da, k[0], db, k[1]) not in CACHE and NMIN[da][k[0]] > 0 and NMIN[db][k[1]] > 0]
    if todo:
        paths = [(XYT[da][a], XYT[db][b]) for a, b in todo]; chunk = max(200, len(paths) // (4 * WORKERS) + 1)
        with ThreadPoolExecutor(WORKERS) as pool: sc = np.concatenate(list(pool.map(lambda s: bozorth3_scores(paths[s:s + chunk]), range(0, len(paths), chunk))))
        for (a, b), v in zip(todo, sc): CACHE[(da, a, db, b)] = v
    for i, (a, b) in enumerate(keys): out[i] = CACHE.get((da, a, db, b), 0.0)       # empty template -> 0
    return out
RESULTS, THRESHOLDS = [], {}
for rep in REPS:
    rd = L.run_dir(KDIR, rep, ARM); proto, P = load_pairs(rep)
    if L.run_complete(rd): S = load_scores(rd); print(rep, 'already scored -> loaded')
    else:
        t = L.timer(); S = {}
        for n, (da, db, _) in PAIR_SETS.items(): S[n] = (np.asarray(P[n]['label'], int), score_pairs(da, db, P[n])); print(rep, n, len(P[n]['label']), 'pairs')
        save_scores(S, rd); LOG.add(step='score', repetition=rep, status='ok', minutes=round(t() / 60, 1), cache=len(CACHE))
    rows, thr = evaluate(S, KERNEL, ARM, rep); RESULTS += rows; THRESHOLDS[rep] = thr
    L.fig_roc_det({n: S[n] for n in VAL_SOURCES + TEST_SETS}, f'BOZORTH3 {rep[:6]}', FIG / 'runs' / rep)
    L.fig_score_distributions({n: S[n] for n in ('DB1_train', 'DB1xDB2_train', 'DB1_test', 'DB2_test', 'DB1xDB2_test', 'FingerDOS_test')}, {'train EER': thr},
                              f'BOZORTH3 {rep[:6]} — score distributions', FIG / 'runs' / rep / 'score_distributions.png')
    if not L.run_complete(rd):
        pd.DataFrame(rows).to_csv(rd / 'metrics.csv', index=False); (rd / 'thresholds.json').write_text(json.dumps({'train_EER': thr, 'rule': THRESHOLD_RULE}, indent=2))
        L.mark_complete(rd, {'kernel': KERNEL, 'arm': ARM, 'repetition': rep, 'threshold_train_EER': thr})
results = pd.DataFrame(RESULTS)

## 5. Results: BOZORTH3 vs the FPF02 image model on the identical pairs

In [ ]:
ref = reference_rows('FPF02 image model', F02DIR)
ALL = pd.concat([results.assign(kernel='FPF04 BOZORTH3')] + ([pd.DataFrame(ref)] if ref else []), ignore_index=True)
MODELS = list(dict.fromkeys(ALL.kernel)); paper, snum = paper_table(ALL, MODELS)
display(paper[paper.kernel == 'FPF04 BOZORTH3'].drop(columns='kernel')); display(paper)
paired = paired_differences(ALL, 'FPF04 BOZORTH3', [m for m in MODELS if m != 'FPF04 BOZORTH3']) if len(MODELS) > 1 else pd.DataFrame()
if len(paired): display(paired[paired.metric.isin(['roc_auc', 'eer', 'f1_macro']) & paired.set.isin(['MOLF test (pooled)', 'FingerDOS_test'])].round(2))

## 6. Time (single thread) and storage

In [ ]:
_, P0 = load_pairs(REPS[0]); p = P0['DB1_test']; ok = [(a, b) for a, b in zip(p['index_a'], p['index_b']) if NMIN['MOLF_DB1'][a] > 0 and NMIN['MOLF_DB1'][b] > 0][:TIMING_PAIRS]
paths = [(XYT['MOLF_DB1'][a], XYT['MOLF_DB1'][b]) for a, b in ok]; bozorth3_scores(paths[:100])
t = L.timer(); bozorth3_scores(paths); per_pair = 1000 * t() / len(paths)
t = L.timer()
for a, b in paths[:SINGLE_CALLS]: bozorth3_scores([(a, b)])
single_call = 1000 * t() / SINGLE_CALLS
T_ROWS = []
for d in L.DATASETS:
    T_ROWS += [{'stage': 'template creation', 'item': f'FingerJetFX extraction per image ({d})', 'ms': EXTRACT_MS[d], 'note': 'FPF00, single thread'},
               {'stage': 'template creation', 'item': f'ISO template -> xyt per template ({d})', 'ms': CONV_MS[d], 'note': 'single thread'},
               {'stage': 'storage', 'item': f'ISO 19794-2 template, mean bytes ({d})', 'ms': np.nan, 'note': f'{TEMPLATE_BYTES[d]:.0f} B'},
               {'stage': 'storage', 'item': f'xyt file, mean bytes ({d})', 'ms': np.nan, 'note': f'{XYT_BYTES[d]:.0f} B'}]
T_ROWS += [{'stage': '1:1 comparison', 'item': f'BOZORTH3 per pair ({len(paths)} pairs in one process)', 'ms': per_pair, 'note': 'single thread, process start-up amortised'},
           {'stage': '1:1 comparison', 'item': 'BOZORTH3 one pair per process call', 'ms': single_call, 'note': 'includes process start-up'},
           {'stage': '1:1 verification', 'item': 'probe image -> decision (MOLF DB1): extraction + xyt + one comparison', 'ms': EXTRACT_MS['MOLF_DB1'] + CONV_MS['MOLF_DB1'] + per_pair,
            'note': 'enrolled template stored; comparison amortised'}]
timing = pd.DataFrame(T_ROWS); timing['cpu'] = ENV.get('cpu_model', ''); display(timing)

## 7. Figures, manifest and workbook

In [ ]:
for ps in VAL_SOURCES + TEST_SETS:
    curves = {'FPF04 BOZORTH3': [load_scores(KDIR / 'runs' / r / ARM)[ps] for r in REPS]}
    if ref:
        w2 = json.loads((F02DIR / 'winner_manifest.json').read_text())['winning_arm']; curves['FPF02 image model'] = [load_scores(F02DIR / 'runs' / r / w2)[ps] for r in REPS]
    L.fig_mean_roc(curves, f'Mean ROC over {len(REPS)} repetitions — {ps}', FIG / f'mean_roc_{ps}.png')
fig_model_bars(snum, MODELS, FIG / 'paper_metrics_bozorth3_vs_image_model.png')
per_rep = {r: {'seed': L.rep_seed(r), 'run_dir': L.rel(ws, KDIR / 'runs' / r / ARM), 'threshold_train_EER': THRESHOLDS[r],
               'files': {'pair_scores': L.rel(ws, KDIR / 'runs' / r / ARM / 'pair_scores.npz')}} for r in REPS}
(KDIR / 'winner_manifest.json').write_text(json.dumps({'kernel': KERNEL, 'kernel_name': KERNEL_NAME, 'winning_arm': ARM, 'config': CONFIG, 'note': 'baseline (single arm)', 'repetitions': per_rep}, indent=2, default=str))
SHEETS = {'Paper Table': paper, 'Paired Differences': paired, 'Summary (numeric)': snum, 'Verification Results': L.to_percent(ALL),
          'Thresholds': pd.DataFrame([{'repetition': r, 'train_EER_threshold': t} for r, t in THRESHOLDS.items()]), 'Time and Storage': timing, 'Configuration': CONFIG,
          'Run Log': LOG.frame(), 'Environment': ENV, 'Figure Index': L.figure_index(KDIR)}
L.write_workbook(KDIR / 'FPF04_BOZORTH3_Results.xlsx', SHEETS)
for name, df in (('paper_table', paper), ('results_per_repetition', L.to_percent(ALL)), ('paired_differences', paired), ('time_and_storage', timing)): df.to_csv(KDIR / f'FPF04_{name}.csv', index=False)
print('FPF04 complete.')